# Spectrogram Calendar from recordings in Google Drive

This notebook runs [**spectro-calendar**](https://github.com/biodiversica/spectro-calendar) over audio stored in your **Google Drive** and builds a browsable **HTML calendar of spectrograms** — one thumbnail per recording, with dates as columns and times of day as rows, so weeks of monitoring can be scanned at a glance.

---

### What this notebook does (in order):
1. **Connects to your Google Drive** to read the recordings
2. **Installs spectro-calendar and ffmpeg** automatically
3. **Scans your audio folder** and shows exactly which recordings will be included
4. **Generates the spectrograms and the HTML calendar**
5. **Opens the calendar inside the notebook**, with working audio players
6. **Packs everything into a `.zip`** to save on your Drive and/or download

### Folder structure:
```
# Flat — all recordings in one folder
MyDrive/audio/
  20250101_180000.WAV
  20250101_183000.WAV

# Subfolders — one folder per recorder (turn on "include subfolders" in Step 2)
MyDrive/audio/
  POINT_A/
    20250101_180000.WAV
  POINT_B/
    20250101_180000.WAV
```

### Filename format:
Every `.wav` file must carry its recording date and time in the filename:
- **AudioMoth** (default): `YYYYMMDD_HHMMSS.WAV` — e.g. `20250615_203000.WAV`
- **With a prefix**: `SM4_20250615_203000.wav` — set the prefix to `SM4_`
- **Any other layout**: set a custom date/time format, e.g. `%Y-%m-%d_%H-%M-%S` for `2025-06-15_20-30-00.wav`

The whole calendar is built from those timestamps, so **a filename that cannot be read stops the run** — the notebook names the offending files in Step 3.

### About the calendar and the audio:
- Google Drive **cannot display an HTML page**: opening `index_.html` there only downloads it. So the finished calendar is delivered as a **`.zip`** — unzip it on your computer and open `index_.html` in a browser.
- The **audio players do work inside this notebook** (Step 5): a small web server running in the Colab runtime streams both the page and the recordings straight from your mounted Drive, without copying anything.
- For players that keep working **after the download**, Step 6 can copy the recordings into the zip. That makes it self-contained but much larger — WAV is uncompressed, and one AudioMoth minute at 48 kHz is roughly 23 MB.

### Before you start:
- A **Google account**, with the recordings in Google Drive
- Only `.wav` / `.WAV` files are read; other formats are ignored

---
## Step 1 — Connect Google Drive & install the software

Run the two cells below. The first asks you to **allow access to your Google Drive** — click the link and follow the steps.

In [ ]:
#@title 📂 Connect Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive connected successfully.")

In [ ]:
#@title 📦 Install spectro-calendar { display-mode: "form" }

#@markdown Installs the [spectro-calendar](https://github.com/biodiversica/spectro-calendar)
#@markdown package and makes sure `ffmpeg` is available (Colab usually ships it already).
#@markdown Run this cell again after a runtime restart.

import shutil
import subprocess
import sys

!pip install -q "git+https://github.com/biodiversica/spectro-calendar.git"

if shutil.which("ffmpeg") is None:
    print("ffmpeg not found, installing it (this takes a minute) ...")
    !apt-get -qq update
    !apt-get -qq install -y ffmpeg

_ffmpeg = shutil.which("ffmpeg")
if _ffmpeg:
    _version = subprocess.run([_ffmpeg, "-version"], capture_output=True, text=True).stdout.splitlines()[0]
    print(f"ffmpeg : {_version}")
else:
    print("ffmpeg is NOT available -- the slower scipy backend will be used instead.")

_check = subprocess.run([sys.executable, "-m", "spectro_calendar", "--help"], capture_output=True, text=True)
if _check.returncode == 0:
    import spectro_calendar
    print(f"spectro-calendar {spectro_calendar.__version__} installed successfully.")
else:
    print("spectro-calendar did not install correctly. The error was:")
    print(_check.stderr[-2000:])

---
## Step 2 — Configuration

Fill in the four forms below. **You do not need to edit any code** — just type or select your values and run each cell, from top to bottom:

1. **Audio input** — where the recordings are and how their filenames are formatted
2. **Date & time filters** — which recordings go into the calendar
3. **Spectrogram settings** — how each thumbnail looks
4. **Output, audio & annotation** — what the run produces

> **Tip:** the forms hide the code automatically. To see the underlying code, click the `{ }` icon in the top-right corner of any form cell.

In [ ]:
#@title 🗂️ Audio input { display-mode: "form" }

from datetime import datetime
from pathlib import Path

#@markdown **Audio folder** — the folder on your Google Drive holding the recordings.
#@markdown Example: `/content/drive/MyDrive/my_project/audio`
DRIVE_INPUT_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

#@markdown **Include subfolders** — also read `.wav` files inside subfolders, e.g. one folder
#@markdown per recorder. Each recording's images are written to a matching subfolder of the output.
RECURSIVE = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Filename prefix** — the literal text before the date/time in the filename,
#@markdown **including** its separator. Leave blank for plain AudioMoth names.
#@markdown Example: `SM4_` for `SM4_20250615_203000.wav`
FILENAME_PREFIX = ""  #@param {type:"string"}

#@markdown **Date/time format** — how the timestamp is written, once the prefix is removed
#@markdown ([strptime codes](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes)).
#@markdown `%Y%m%d_%H%M%S` matches `20250615_203000`, `%Y-%m-%d_%H-%M-%S` matches `2025-06-15_20-30-00`.
DATETIME_FORMAT = "%Y%m%d_%H%M%S"  #@param {type:"string"}

REC_DIR = Path(DRIVE_INPUT_DIR.strip().rstrip("/"))
if not REC_DIR.is_dir():
    raise FileNotFoundError(f"Audio folder not found (check the path in this form): {REC_DIR}")

_example = FILENAME_PREFIX + datetime(2025, 6, 15, 20, 30, 0).strftime(DATETIME_FORMAT) + ".WAV"

print(f"Audio folder      : {REC_DIR}")
print(f"Subfolders        : {RECURSIVE}")
print(f"Expected filename : {_example}")

In [ ]:
#@title 📅 Date & time filters { display-mode: "form" }

#@markdown Every filter is optional — with the defaults, **all** recordings go into the calendar.
#@markdown These filters are the main way to keep a run short: each selected file is read in full
#@markdown from your Drive.

#@markdown ---
#@markdown **Date range** — `YYYYMMDD`. Leave an end blank for no limit on that side.
START_DATE = ""  #@param {type:"string"}
END_DATE = ""  #@param {type:"string"}

#@markdown **Specific dates** (optional) — a comma-separated `YYYYMMDD` list, e.g. `20250615, 20250617`.
#@markdown When filled, it **replaces** the date range above (the two cannot be combined).
DATES = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Daily time window** — `HHMMSS`. Only recordings made between these two times
#@markdown of day are included, on every selected date.
START_TIME = "000000"  #@param {type:"string"}
END_TIME = "235900"  #@param {type:"string"}

#@markdown **Time step (minutes)** — keep only one recording per interval per day; `0` keeps all.
#@markdown E.g. `30` turns a recording every 5 minutes into one every 30.
TIME_STEP = 0  #@param {type:"integer"}

START_DATE = START_DATE.strip()
END_DATE = END_DATE.strip()
START_TIME = START_TIME.strip() or "000000"
END_TIME = END_TIME.strip() or "235900"
DATES_LIST = [d.strip() for d in DATES.replace(";", ",").split(",") if d.strip()]
TIME_STEP = max(0, int(TIME_STEP))

if DATES_LIST and (START_DATE or END_DATE):
    raise ValueError("Use either 'specific dates' or a start/end date range, not both.")
for _value in DATES_LIST + [d for d in (START_DATE, END_DATE) if d]:
    if not (_value.isdigit() and len(_value) == 8):
        raise ValueError(f"Dates must be written as YYYYMMDD, got: {_value}")
for _value in (START_TIME, END_TIME):
    if not (_value.isdigit() and len(_value) == 6):
        raise ValueError(f"Times must be written as HHMMSS, got: {_value}")

if DATES_LIST:
    print(f"Dates       : {', '.join(DATES_LIST)}")
else:
    print(f"Date range  : {START_DATE or '...'} -> {END_DATE or '...'}")
print(f"Time window : {START_TIME} -> {END_TIME}")
print(f"Time step   : {TIME_STEP or '-'}")

In [ ]:
#@title 🎨 Spectrogram settings { display-mode: "form" }

#@markdown **Use ffmpeg** — the fast spectrogram backend, and the only one that honours the
#@markdown colour, gain and frequency-scale settings below. If ffmpeg is missing, the tool falls
#@markdown back to scipy automatically (slower, always the `plasma` palette on a linear scale).
USE_FFMPEG = True  #@param {type:"boolean"}

#@markdown ---
#@markdown **Frequency range (Hz)** — the vertical axis of every spectrogram. Narrow it to the
#@markdown band your target sounds live in and the thumbnails become far easier to read.
LOWEST_FREQ = 0  #@param {type:"integer"}
HIGHEST_FREQ = 20000  #@param {type:"integer"}

#@markdown **Frequency scale** — `log` gives low frequencies more room (ffmpeg only).
FREQ_SCALE = "lin"  #@param ["lin", "log"]

#@markdown ---
#@markdown **Gain (dB)** — brightens the spectrogram; raise it for quiet recordings (ffmpeg only).
GAIN = 1  #@param {type:"integer"}

#@markdown **Gain scale** — how magnitudes are mapped to colour (ffmpeg only).
GAIN_SCALE = "log"  #@param ["log", "sqrt", "cbrt", "4thrt", "5thrt", "lin"]

#@markdown **Colour palette** — the spectrogram colours (ffmpeg only).
COLOR_CHOICE = "plasma"  #@param ["plasma", "viridis", "magma", "cividis", "intensity", "rainbow", "moreland", "nebulae", "fire", "fiery", "fruit", "cool", "green", "terrain", "channel"]

#@markdown ---
#@markdown **Full-size image** (`width x height`) and **thumbnail** (`width:height`), in pixels.
#@markdown The thumbnail is downscaled from the full-size render, so the first value also affects
#@markdown how much detail a thumbnail can show. The thumbnail size is the size of a calendar cell —
#@markdown larger cells mean a clearer but heavier page.
IMG_SIZE = "1080x720"  #@param {type:"string"}
THUMBNAIL_SCALE = "108:72"  #@param {type:"string"}

#@markdown **Keep the full-size images** — also save each full-resolution spectrogram next to its
#@markdown thumbnail, to open when a cell needs a closer look. Off by default: the calendar only
#@markdown uses thumbnails, and full-size PNGs take much more space in the zip.
SAVE_FULLSIZE = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Spectrogram label** (optional) — a suffix that keeps several runs apart inside the same
#@markdown calendar, e.g. `lf` and `hf` for a low- and a high-frequency band. It names the page
#@markdown (`index_lf.html`) and every image it contains.
SPEC_LABEL = ""  #@param {type:"string"}

SPEC_LABEL = SPEC_LABEL.strip()

print(f"Backend         : {'ffmpeg' if USE_FFMPEG else 'scipy'}")
print(f"Frequency range : {LOWEST_FREQ} - {HIGHEST_FREQ} Hz ({FREQ_SCALE})")
print(f"Thumbnail       : {THUMBNAIL_SCALE.replace(':', ' x ')} px")
print(f"Label           : {SPEC_LABEL or '-'}")

In [ ]:
#@title 📦 Output, audio & annotation { display-mode: "form" }

from pathlib import Path

#@markdown **Calendar name** — names the folder holding the generated images, the HTML page and
#@markdown the stylesheet, and the zip built from it in Step 6.
CALENDAR_NAME = "my_calendar"  #@param {type:"string"}

#@markdown ---
#@markdown **Audio players** — add a player under each thumbnail. They stream from your Drive in
#@markdown the notebook preview (Step 5); to keep them working after the download, bundle the audio
#@markdown into the zip in Step 6.
INCLUDE_AUDIO = True  #@param {type:"boolean"}

#@markdown **Annotation mode** — add a checkbox under each thumbnail plus a toolbar that exports the
#@markdown ticked recordings as a `.txt` list. `Shift+click` a second box to select the whole range
#@markdown between them.
ANNOTATE = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Start from scratch** — delete the images of a previous run of this same calendar before
#@markdown generating. Leave it off to resume an interrupted run: with the ffmpeg backend,
#@markdown recordings whose thumbnail already exists are skipped.
CLEAR = False  #@param {type:"boolean"}

#@markdown ---
#@markdown Everything is generated on the runtime's local disk, which is fast but temporary.
#@markdown Step 6 is what puts the result on your Drive.

CALENDAR_NAME = (CALENDAR_NAME.strip() or "my_calendar").replace(" ", "_").replace("/", "-")
OUT_DIR = Path("/content/spectro_calendar") / CALENDAR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working folder  : {OUT_DIR}")
print(f"Audio players   : {INCLUDE_AUDIO}")
print(f"Annotation mode : {ANNOTATE}")

---
## Step 3 — Scan the recordings

This cell lists the `.wav` files in your audio folder, reads the timestamp out of every filename and applies the filters from Step 2, so you can see **exactly what the calendar will contain before anything is generated**.

It also reports how much audio has to be read from Drive. That read is the slow part of the whole process — if the number looks large, go back to Step 2 and narrow the date range, the daily time window or the time step.

In [ ]:
#@title 🔍 Scan { display-mode: "form" }

from spectro_calendar.cli import (
    filter_dates_by_range,
    filter_wav_files_by_time_step,
    filter_wav_files_by_time_window,
    get_available_dates,
    parse_recording_datetime,
    parse_time_string,
)

_candidates = REC_DIR.rglob("*") if RECURSIVE else REC_DIR.iterdir()
ALL_WAVS = [p for p in _candidates if p.is_file() and p.suffix.lower() == ".wav"]
if not ALL_WAVS:
    raise FileNotFoundError("No .wav files found in the audio folder. If the recordings are inside subfolders, turn on 'include subfolders' in Step 2.")

FILE_DATES = {}
_unreadable = []
for _wav in ALL_WAVS:
    try:
        FILE_DATES[_wav] = parse_recording_datetime(_wav, DATETIME_FORMAT, FILENAME_PREFIX)
    except ValueError:
        _unreadable.append(_wav)

if _unreadable:
    print("The date and time could not be read from these filenames:")
    for _wav in _unreadable[:10]:
        print(f"   {_wav.name}")
    if len(_unreadable) > 10:
        print(f"   ... (+{len(_unreadable) - 10})")
    raise ValueError("Adjust the filename prefix and/or the date/time format in Step 2, or move these files out of the folder.")

ALL_WAVS.sort(key=lambda w: FILE_DATES[w])
_available = get_available_dates(FILE_DATES)

if DATES_LIST:
    _missing = [d for d in DATES_LIST if d not in _available]
    if _missing:
        raise ValueError(f"No recordings on the requested date(s): {', '.join(_missing)}")
    _dates = DATES_LIST
elif START_DATE or END_DATE:
    _dates = filter_dates_by_range(_available, START_DATE or None, END_DATE or None)
else:
    _dates = _available

SELECTED_WAVS = filter_wav_files_by_time_window(
    ALL_WAVS,
    FILE_DATES,
    parse_time_string(START_TIME).time(),
    parse_time_string(END_TIME).time(),
    _dates,
)
if TIME_STEP:
    SELECTED_WAVS = filter_wav_files_by_time_step(SELECTED_WAVS, FILE_DATES, _dates, TIME_STEP)
if not SELECTED_WAVS:
    raise ValueError("No recording matches the filters from Step 2. Widen the date range, the time window or the time step.")

# One calendar cell per date and time of day, exactly as the table is built.
_cells = {(FILE_DATES[w].strftime("%Y%m%d"), FILE_DATES[w].strftime("%H%M%S")) for w in SELECTED_WAVS}
_columns = sorted({cell[0] for cell in _cells})
_rows = sorted({cell[1] for cell in _cells})
_bytes = sum(w.stat().st_size for w in SELECTED_WAVS)

print(f"Recordings found : {len(ALL_WAVS)}")
print(f"Selected         : {len(SELECTED_WAVS)}")
print(f"Dates            : {_columns[0]} -> {_columns[-1]}  ({len(_columns)} day(s))")
print(f"Times of day     : {_rows[0][:2]}:{_rows[0][2:4]} -> {_rows[-1][:2]}:{_rows[-1][2:4]}  ({len(_rows)})")
print(f"Calendar table   : {len(_columns)} column(s) x {len(_rows)} row(s)")
print(f"Audio to read    : {_bytes / 2 ** 30:.2f} GB")

if len(SELECTED_WAVS) > len(_cells):
    print(f"\n{len(SELECTED_WAVS) - len(_cells)} recording(s) share a date and time with another one; the calendar has a single cell per slot, so only one of each pair is shown (all thumbnails are still generated).")

---
## Step 4 — Generate the calendar

This is the main step: every selected recording is read from Drive, turned into a spectrogram thumbnail, and laid out in the HTML calendar.

Everything is written to the **runtime's local disk** (`/content/spectro_calendar/...`), not directly to your Drive — writing thousands of small PNGs over Drive is far slower. Step 6 packs the result into a single zip file, which is what goes to your Drive.

> Depending on how many recordings were selected, this step can take a while. Progress is shown below.
>
> If the run is interrupted (or the Colab session drops), just run it again with **start from scratch** off: with the ffmpeg backend, recordings whose thumbnail already exists are skipped, so the run picks up where it stopped.

In [ ]:
#@title 🚀 Generate the calendar { display-mode: "form" }

#@markdown **Show every file** — print one line per recording instead of a running count.
#@markdown Useful to see what is happening on a small run; very noisy on a large one.
SHOW_EVERY_FILE = False  #@param {type:"boolean"}

import os
import re
import shlex
import subprocess
import sys
import time

cmd = [sys.executable, "-m", "spectro_calendar", str(REC_DIR), "--output-dir", str(OUT_DIR)]
if RECURSIVE:
    cmd.append("--recursive")
if FILENAME_PREFIX:
    cmd += ["--filename-prefix", FILENAME_PREFIX]
cmd += ["--datetime-format", DATETIME_FORMAT]
if DATES_LIST:
    cmd += ["--dates", *DATES_LIST]
else:
    if START_DATE:
        cmd += ["--start-date", START_DATE]
    if END_DATE:
        cmd += ["--end-date", END_DATE]
cmd += ["--start-time", START_TIME, "--end-time", END_TIME]
if TIME_STEP:
    cmd += ["--time-step", str(TIME_STEP)]
if USE_FFMPEG:
    cmd.append("--use-ffmpeg")
cmd += [
    # One recording at a time: parallel workers quickly exhaust the RAM
    # of a Colab runtime and crash the session.
    "--max-cores", "1",
    "--lowest-freq", str(LOWEST_FREQ),
    "--highest-freq", str(HIGHEST_FREQ),
    "--freq-scale", FREQ_SCALE,
    "--gain", str(GAIN),
    "--gain-scale", GAIN_SCALE,
    "--color-choice", COLOR_CHOICE,
    "--img-size", IMG_SIZE,
    "--thumbnail-scale", THUMBNAIL_SCALE,
]
if SPEC_LABEL:
    cmd += ["--spec-label", SPEC_LABEL]
if SAVE_FULLSIZE:
    cmd.append("--save-fullsize")
if CLEAR:
    cmd.append("--clear")
if INCLUDE_AUDIO:
    cmd.append("--include-audio")
if ANNOTATE:
    cmd.append("--annotate")

print(f"Running:\n  spectro-calendar {shlex.join(cmd[3:])}\n")

# One line per file is far too much output for a month of recordings; count
# those lines instead and keep everything else (warnings, errors) visible.
_per_file = re.compile(r"^(Processing |making (thumbnail|fullsize) spectrogram for|Spectrogram for )")
_finished = re.compile(r"^(making thumbnail spectrogram for|Spectrogram for )")
_done = 0
_start = time.time()
_process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
_counting = False  # the running count is on screen, with no line break yet
for _line in _process.stdout:
    if SHOW_EVERY_FILE or not _per_file.match(_line):
        # A warning (e.g. an unreadable recording) goes on a line of its own
        print(("\n" if _counting else "") + _line, end="")
        _counting = False
    elif _finished.match(_line):
        _done += 1
        print(f"\r   {_done}/{len(SELECTED_WAVS)} spectrograms done   ", end="", flush=True)
        _counting = True
_process.wait()
if _counting:
    print()

if _process.returncode != 0:
    raise RuntimeError(f"spectro-calendar failed -- see the message above (exit code {_process.returncode})")

INDEX_PAGES = sorted(OUT_DIR.glob("index_*.html"))
print(f"Total time     : {time.time() - _start:.0f} s")
print(f"Images written : {len(list(OUT_DIR.rglob('*.png')))}")
for _page in INDEX_PAGES:
    print(f"Calendar page  : {_page}")

---
## Step 5 — Open the calendar inside the notebook

The cell below starts a small web server inside the Colab runtime and shows the calendar right here, in a frame. The **Full screen** button above it gives a large calendar the whole screen (Colab cannot open it in a separate browser tab: the preview address only works inside the notebook).

This is the one place where **the audio players work without copying anything**: each recording is read from your mounted Drive, on demand, only when you press play (a long recording takes a moment to load). The same goes for annotation mode — ticking boxes, `Shift+click` ranges and the `.txt` export all work here, and the selection is remembered between reloads.

The preview only works while **this runtime is alive** and only for your own Google account. Once the session ends, use the zip from Step 6.

In [ ]:
#@title 👀 Open the calendar { display-mode: "form" }

#@markdown Starts a web server inside the Colab runtime and shows the calendar below.
#@markdown The audio players stream straight from your Drive — nothing is copied.

#@markdown **Frame height** (pixels) — how tall the preview below is.
PREVIEW_HEIGHT = 700  #@param {type:"integer"}

#@markdown **Port** — only worth changing if this one is already taken in the runtime.
PREVIEW_PORT = 8765  #@param {type:"integer"}

import functools
import os
import re
import threading
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer

from IPython.display import HTML, display


AUDIO_SCRIPT = """\
// Colab's proxy passes the page's own requests (the thumbnails load fine) but
// not the browser's media requests, so an <audio> pointed straight at a WAV
// never starts. Fetch the recording as an ordinary request and play it from
// memory instead, keeping only the last one played in memory.
(function () {
  var loaded = null;

  function note(audio, text) {
    var label = audio.nextElementSibling;
    if (!label || !label.classList.contains("audio-status")) {
      label = document.createElement("div");
      label.className = "audio-status";
      label.style.cssText = "font-size:11px;color:#666";
      audio.insertAdjacentElement("afterend", label);
    }
    label.textContent = text;
  }

  function release(audio) {
    audio.pause();
    URL.revokeObjectURL(audio.dataset.blob);
    delete audio.dataset.blob;
    audio.removeAttribute("src");
    audio.load();
  }

  document.querySelectorAll("audio").forEach(function (audio) {
    var source = audio.querySelector("source");
    if (!source) return;
    var url = source.src;
    audio.addEventListener("play", function () {
      if (audio.dataset.blob) return;
      audio.pause();
      if (audio.dataset.busy) return;
      audio.dataset.busy = "1";
      note(audio, "loading...");
      fetch(url).then(function (response) {
        if (!response.ok) throw new Error("HTTP " + response.status);
        return response.blob();
      }).then(function (blob) {
        if (loaded && loaded !== audio) release(loaded);
        loaded = audio;
        audio.dataset.blob = URL.createObjectURL(new Blob([blob], {type: "audio/wav"}));
        audio.src = audio.dataset.blob;
        note(audio, "");
        return audio.play().catch(function () { note(audio, "ready, press play"); });
      }).catch(function (error) {
        note(audio, "could not load the recording (" + error.message + ")");
      }).finally(function () {
        delete audio.dataset.busy;
      });
    });
  });
})();
"""


class CalendarHandler(SimpleHTTPRequestHandler):
    """Static file server that understands Range requests, so the browser can
    seek inside a WAV instead of having to download it whole before playing."""

    def log_message(self, *args):
        pass  # keep the notebook output clean

    def end_headers(self):
        self.send_header("Accept-Ranges", "bytes")
        super().end_headers()

    def send_calendar_page(self, path):
        """Serves a calendar page with AUDIO_SCRIPT added. Only the served copy
        changes: the file on disk, and so the zip from Step 6, stays as it is."""
        with open(path, encoding="utf-8") as source:
            page = source.read()
        if INCLUDE_AUDIO:
            page = page.replace("</body>", f"<script>\n{AUDIO_SCRIPT}</script>\n</body>", 1)
        body = page.encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        path = self.translate_path(self.path)
        if os.path.basename(path).startswith("index_") and path.endswith(".html") and os.path.isfile(path):
            return self.send_calendar_page(path)

        _range = self.headers.get("Range", "")
        wanted = re.fullmatch(r"bytes=(\d*)-(\d*)", _range.strip()) if _range else None
        if not wanted or not os.path.isfile(path):
            return super().do_GET()

        size = os.path.getsize(path)
        first, last = wanted.group(1), wanted.group(2)
        if first == "":
            start, end = max(0, size - int(last or 0)), size - 1
        else:
            start, end = int(first), int(last) if last else size - 1
        end = min(end, size - 1)
        if start > end or start >= size:
            self.send_response(416)
            self.send_header("Content-Range", f"bytes */{size}")
            self.end_headers()
            return

        self.send_response(206)
        self.send_header("Content-Type", self.guess_type(path))
        self.send_header("Content-Range", f"bytes {start}-{end}/{size}")
        self.send_header("Content-Length", str(end - start + 1))
        self.end_headers()
        try:
            with open(path, "rb") as source:
                source.seek(start)
                remaining = end - start + 1
                while remaining > 0:
                    chunk = source.read(min(256 * 1024, remaining))
                    if not chunk:
                        break
                    self.wfile.write(chunk)
                    remaining -= len(chunk)
        except (BrokenPipeError, ConnectionResetError):
            pass  # the browser hung up (player closed, page reloaded)


INDEX_PAGES = sorted(OUT_DIR.glob("index_*.html"))
if not INDEX_PAGES:
    raise FileNotFoundError("No calendar page found -- run Step 4 first.")

# The page links to the recordings by a relative path that climbs out of the
# calendar folder, so the server has to be rooted at the first directory
# holding both the calendar and the audio -- usually /content.
SERVE_ROOT = os.path.commonpath([str(OUT_DIR.resolve()), str(REC_DIR.resolve())])

_running = globals().get("PREVIEW_SERVER")
if _running is not None:
    _running.shutdown()
    _running.server_close()

PREVIEW_SERVER = ThreadingHTTPServer(("", PREVIEW_PORT), functools.partial(CalendarHandler, directory=SERVE_ROOT))
PREVIEW_SERVER.daemon_threads = True
threading.Thread(target=PREVIEW_SERVER.serve_forever, daemon=True).start()

try:
    from google.colab.output import eval_js

    _base = eval_js(f"google.colab.kernel.proxyPort({PREVIEW_PORT})").rstrip("/")
    _urls = [f"{_base}/{os.path.relpath(page, SERVE_ROOT)}" for page in INDEX_PAGES]
    # Colab's preview address only works inside this output frame (browser
    # storage partitioning), so a link to a new tab would just give a 404.
    # Full screen stands in for it; where the browser refuses full screen
    # inside Colab, the frame grows to the height of the screen instead.
    _buttons = [
        '<button onclick="var f=document.getElementById(\'calendar-frame\');'
        ' (f.requestFullscreen ? f.requestFullscreen() : Promise.reject())'
        '.catch(function () { f.height = screen.availHeight - 120; });">⛶ Full screen</button>'
    ]
    if len(INDEX_PAGES) > 1:
        _buttons += [
            f'<button onclick="document.getElementById(\'calendar-frame\').src=\'{url}\'">{page.name}</button>'
            for url, page in zip(_urls, INDEX_PAGES)
        ]
    display(HTML(
        f'<p>{" ".join(_buttons)}</p>'
        f'<iframe id="calendar-frame" src="{_urls[0]}" width="100%" height="{PREVIEW_HEIGHT}"'
        ' allow="fullscreen; autoplay" allowfullscreen'
        ' style="border:1px solid #bbb;border-radius:4px"></iframe>'
    ))
except Exception as error:
    print(f"Could not open the preview here (this cell only works in Google Colab). The calendar is on the runtime's disk at:\n  {INDEX_PAGES[0]}\n  ({error})")

---
## Step 6 — Pack it into a zip, save and download

The cell below packs the calendar (thumbnails, `index_*.html` and the stylesheet) into a single `.zip`, copies it to your Google Drive and/or downloads it to your computer.

Unzip it locally and open `index_*.html` in a browser.

**What happens to the audio players in the zip?**
- **Off (default):** the players are removed from the zipped page — they would point at your Drive and show up as broken controls. The calendar itself, with all its thumbnails, works offline.
- **On:** every recording used by the calendar is copied into the zip under `audio/`, and the links are rewritten to point there. The calendar is then fully self-contained, but the zip grows by the size of all that audio — check the "audio to read" figure from Step 3 before turning this on.

In [ ]:
#@title 📦 Pack, save and download { display-mode: "form" }

#@markdown **Bundle the audio in the zip** — copy every recording used by the calendar into the zip,
#@markdown so the players keep working offline. This adds the full size of that audio to the zip
#@markdown (see "audio to read" in Step 3). With this off, the players are removed from the zipped
#@markdown page, since they would only point back at your Drive.
BUNDLE_AUDIO_IN_ZIP = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Save the zip to Google Drive** — and the folder it goes to (created if needed).
SAVE_ZIP_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/spectro_calendars"  #@param {type:"string"}

#@markdown **Download the zip** — sends it to your computer through the browser. For a large zip,
#@markdown saving to Drive and downloading from there is more reliable.
DOWNLOAD_ZIP = True  #@param {type:"boolean"}

import html as html_module
import os
import re
import shutil
import zipfile
from pathlib import Path

_audio_element = re.compile(r"<audio\b.*?</audio>", re.S)
_audio_source = re.compile(r'(<source src=")([^"]+)(" type="audio/wav">)')
BUNDLED_AUDIO = {}


def prepare_page(text):
    """Rewrites the audio links for the offline copy: either pointing them at the
    recordings bundled into the zip, or removing the players altogether."""
    if not INCLUDE_AUDIO:
        return text
    if not BUNDLE_AUDIO_IN_ZIP:
        return _audio_element.sub("", text)

    def rewrite(match):
        wav = Path(os.path.normpath(OUT_DIR / html_module.unescape(match.group(2))))
        inside_zip = "audio/" + Path(os.path.relpath(wav, REC_DIR)).as_posix()
        BUNDLED_AUDIO[wav] = inside_zip
        return match.group(1) + html_module.escape(inside_zip, quote=True) + match.group(3)

    return _audio_source.sub(rewrite, text)


ZIP_PATH = Path("/content") / f"{CALENDAR_NAME}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for _file in sorted(OUT_DIR.rglob("*")):
        if not _file.is_file():
            continue
        _arcname = f"{CALENDAR_NAME}/{_file.relative_to(OUT_DIR).as_posix()}"
        if _file.parent == OUT_DIR and _file.name.startswith("index_") and _file.suffix == ".html":
            archive.writestr(_arcname, prepare_page(_file.read_text()))
        else:
            archive.write(_file, _arcname)
    # WAV audio barely compresses, so store it as-is rather than spend the time.
    for _wav, _inside_zip in BUNDLED_AUDIO.items():
        if _wav.exists():
            archive.write(_wav, f"{CALENDAR_NAME}/{_inside_zip}", compress_type=zipfile.ZIP_STORED)
        else:
            print(f"Recording not found, skipped: {_wav}")

_size = ZIP_PATH.stat().st_size
print(f"Zip file       : {ZIP_PATH}  ({_size / 2 ** 20:.1f} MB)")
if BUNDLED_AUDIO:
    print(f"Recordings bundled in the zip : {len(BUNDLED_AUDIO)}")
elif INCLUDE_AUDIO:
    print("The audio players were removed from the zipped page (turn on 'bundle the audio in the zip' to keep them).")

if SAVE_ZIP_TO_DRIVE:
    _drive_dir = Path(DRIVE_OUTPUT_DIR.strip().rstrip("/"))
    _drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_PATH, _drive_dir / ZIP_PATH.name)
    print(f"Saved to Drive : {_drive_dir / ZIP_PATH.name}")

print("Unzip it and open the index_*.html file in a browser.")

if DOWNLOAD_ZIP:
    if _size > 2 * 2 ** 30:
        print("The zip is larger than 2 GB -- download it from your Google Drive instead of through the browser.")
    else:
        from google.colab import files

        files.download(str(ZIP_PATH))

---
## Tips & troubleshooting

**The run is slow.** Nearly all the time goes into reading WAV files from Google Drive, not into computing the spectrograms. The filters in Step 2 are the lever: a time step of 30 or 60 minutes, a narrower daily window (dawn chorus only, say) or a shorter date range cut the transfer proportionally. Keeping **use ffmpeg** on also helps. Recordings are processed one at a time on purpose: running several at once overloads the runtime's RAM and crashes the session.

**Two frequency bands in the same calendar folder.** Run Steps 4–6 twice with the same *calendar name* but a different *spectrogram label* (e.g. `lf` with a 0–2000 Hz range, then `hf` with 2000–20000 Hz). You get `index_lf.html` and `index_hf.html` side by side, and the zip carries both.

**The session times out on long runs.** Colab disconnects idle notebooks. Re-run Step 4 with **start from scratch** off and the ffmpeg backend: existing thumbnails are skipped, so each run advances a bit further. Keeping the browser tab open and visible helps.

**Annotations after the download.** The `.txt` export works from the downloaded copy too. One caveat: Chrome refuses local storage to pages opened from disk (`file://`), so there the ticked boxes survive only while the tab is open — export before closing it. Firefox keeps them, and the notebook preview (Step 5) keeps them as well.

**`No WAV files found`.** Check the folder path, and turn on **include subfolders** if the recordings sit one level down. Only `.wav`/`.WAV` is read — `.flac` and `.mp3` are ignored.

**A filename could not be read.** The prefix and the date/time format must describe the filename exactly: `SM4_20250615_203000.wav` needs the prefix `SM4_` (with the underscore) and the format `%Y%m%d_%H%M%S`. Step 3 prints an example built from your settings — compare it with a real filename.

**Nothing shows up in Step 5.** The preview needs third-party cookies enabled for `googleusercontent.com`; if the frame stays blank, allow them and run the cell again. **A player says it could not load the recording:** the message carries the reason; `HTTP 404` means the recording is no longer at the path it had in Step 4 (rerun Steps 4 and 5).